# **SLT + RL - Train ĐA DẠNG (chọn thí nghiệm + chọn % data qua 1 cell config)**

Notebook train linh hoạt: **1 cell config** để chọn *train cái gì* và *bao nhiêu % data*, rồi **1
cell chạy**. Hai cách chọn phạm vi:

- **`SCOPE="matrix"`** → gọi `run_all.py --groups ...`: chọn 1 hay nhiều **nhóm** thí nghiệm
  (`core, encoders, algos, ablations, reward, latency`) hoặc `all`. Resumable, tự sinh bảng/hình.
- **`SCOPE="select"`** → gọi `train_select.py`: chọn **1 encoder / 1 algo** cụ thể (nhẹ, hợp cho
  100% chạy dần từng kiến trúc qua nhiều phiên).

Không còn nhánh gloss/P7 và RL-ngoài-decoder - xem `docs/2_Huong_Phat_Trien.md`.

**Setup:** Accelerator = **GPU T4**, Internet = **ON**. Add Input = dataset **`phoenix-poses`**
(pose đã extract). Code lấy qua `git clone` ở Cell 3.

**Quy trình:** Cell 1 (config) → Cell 2–4 (deps, clone, trỏ pose) → Cell 5 (chạy) → Cell 6 (bảng)
→ Cell 7 (đóng gói tải về). Phiên sau đổi config, lặp lại; Cell 8 gộp mọi phiên thành 1 bảng+hình.

> **% data:** `SUBSET=0.05` là mức báo cáo CHÍNH (train 5%, **dev/test luôn full**). 0.25/0.5/1.0
> chạy thêm khi có quota - epoch KHÔNG giảm theo subset nên thời gian ~tỉ lệ với % data.

## **Cell 1 — Cấu hình: chọn % data + phạm vi thí nghiệm**

In [ ]:
# ============================ CẤU HÌNH ============================
# Chỉ cần sửa các biến trong khối này rồi chạy Cell 5. Mọi option liệt kê đầy đủ ở comment.

# ----- 1) % DATA -----
SUBSET = 0.05
#   0.05  = mức BÁO CÁO CHÍNH (train 5% split train)   -> toàn ma trận ~6-9h (gọn 1 session)
#   0.10  = test thêm số liệu                           -> ~gấp đôi 5%
#   0.25  = chạy thêm khi có quota                      -> ~20-25h
#   0.5 / 1.0 = mở rộng / full                          -> rất lớn, trải nhiều session
#   LƯU Ý: dev/test LUÔN full ở mọi mức. Các mức LỒNG NHAU theo seed 42 (5% ⊂ 10% ⊂ 25% ...) nên
#          pose của 10% phải được extract sẵn (đã gồm 5%). Epoch KHÔNG giảm theo subset.

# ----- 2) PHẠM VI: chọn 1 trong 2 cách -----
SCOPE = "matrix"
#   "matrix" = gọi run_all.py --groups  -> chạy theo NHÓM thí nghiệm (dùng GROUPS ở mục 3). Resumable,
#              tự sinh bảng+hình ở cuối. Hợp khi muốn chạy nhiều thứ 1 lệnh.
#   "select" = gọi train_select.py      -> chạy HẸP: 1 encoder / 1 algo (dùng mục 4). Nhẹ, hợp 100%
#              chạy dần từng kiến trúc qua nhiều phiên.

# ----- 3) (khi SCOPE="matrix") CHỌN NHÓM: 1 hay NHIỀU nhóm, phẩy ngăn cách, hoặc "all" -----
GROUPS = "all"
#   Các nhóm hợp lệ:
#     core      = Transformer: XE(80ep) + SCST(20ep)          -> tạo best_xe.pt (Exp 1, có XE baseline)
#     encoders  = 5 encoder còn lại × (XE + SCST)             -> stgcn/gcn/graph_transformer/tcn/perceiver (Exp 4)
#     algos     = PPO / MRT / RAML / DPO trên Transformer      -> tái dùng best_xe.pt của core (Exp 7)
#     ablations = REINFORCE(no-baseline) / A2C(no-clip) / Curriculum -> tái dùng best_xe.pt của core
#     reward    = 4 reward combo: rw_bleu_only/rw_default/rw_len_only/rw_both (Exp 9) — TỰ quét, BỎ QUA mục 5
#     latency   = đo tốc độ + params 6 encoder                -> cần best_xe.pt của encoder đó
#   Ví dụ điền: "all" | "core" | "core,encoders" | "core,algos,reward" | "core,encoders,latency"
#   PHỤ THUỘC: algos/ablations/reward CẦN core chạy trước (best_xe.pt); latency CẦN encoders trước.
#     Cứ để "all" là nó tự lo đúng thứ tự. Nếu tách nhóm, nhớ chạy "core" (và "encoders") trước.

# ----- 4) (khi SCOPE="select") CHỌN 1 CẤU HÌNH CỤ THỂ -----
SELECT_MODE = "single"
#   "single"        = 1 encoder × 1 algo   (XE -> RL -> eval)
#   "encoder_allrl" = 1 encoder × 5 RL     (XE train 1 LẦN, 5 RL scst/ppo/mrt/raml/dpo dùng chung best_xe)
#   "rl_allenc"     = 1 algo   × 6 encoder (mỗi encoder tự train XE + algo đó)
ENCODER = "transformer"
#   transformer | stgcn | gcn | graph_transformer | tcn | perceiver
#   (dùng cho "single" và "encoder_allrl"; "rl_allenc" bỏ qua biến này vì chạy cả 6)
ALGO = "scst"
#   scst | ppo | mrt | raml | dpo   <-- CHỌN THUẬT TOÁN RL Ở ĐÂY
#   (dùng cho "single" và "rl_allenc"; "encoder_allrl" bỏ qua vì chạy cả 5)
TAG = "run1"
#   Nhãn thư mục log (run1_{encoder}_subset{pct}). GIỮ NGUYÊN "run1" để mọi phiên gộp chung 1 bảng.

# ----- 5) REWARD (chỉ 4 loại reward cũ: BLEU + rep/len penalty; KHÔNG dùng BERTScore) -----
#   R = w_bleu·BLEU − w_rep·rep_penalty − w_len·len_penalty
#   4 combo = bật/tắt (w_rep, w_len) ∈ {0, 0.5}:
#     bleu_only(0,0) · default(0.5,0) · len_only(0,0.5) · both(0.5,0.5)
REWARD_BLEU = 1.0   # w_bleu: trọng số BLEU         (thường giữ 1.0)
REWARD_REP  = 0.5   # w_rep : phạt lặp n-gram       (0 hoặc 0.5; số 5%: rep≈0 nên penalty này ít tác dụng)
REWARD_LEN  = 0.0   # w_len : phạt câu ngắn hơn ref (0 hoặc 0.5; số 5%: để 0 tốt hơn — câu vốn đã quá ngắn)
#   LƯU Ý: nếu SCOPE="matrix" và GROUPS chứa "reward", nhóm đó TỰ quét cả 4 combo và BỎ QUA 3 biến trên.
#          3 biến này áp cho MỌI run RL còn lại (core/algos/ablations, và mọi run của "select").

# =================================================================
print(f"SUBSET={SUBSET}  SCOPE={SCOPE}")
print(f"  matrix -> GROUPS={GROUPS}")
print(f"  select -> mode={SELECT_MODE} encoder={ENCODER} algo={ALGO} tag={TAG}")
print(f"  reward -> w_bleu={REWARD_BLEU} w_rep={REWARD_REP} w_len={REWARD_LEN}")

## **Cell 2 — Dependencies**
`einops` không được dùng trong code nên bỏ; `bert-score` chỉ cần nếu bật reward BERTScore (weight>0).

In [ ]:
!pip install -q sentencepiece sacrebleu bert-score

## **Cell 3 - Lấy code từ GitHub**

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Sign-Language-REL_code"])
subprocess.run(["git", "clone", "https://github.com/linhxm/Sign-Language-REL_code.git"], check=True)
os.chdir("/kaggle/working/Sign-Language-REL_code")
print("cwd:", os.getcwd())

## **Cell 4 - Trỏ pose_cache_dir vào dataset pose đã mount + kiểm tra > 0**

In [ ]:
import os, glob
POSE_DIR = "/kaggle/input/phoenix-poses"
if not glob.glob(os.path.join(POSE_DIR, "*.npz")):
    hits = glob.glob("/kaggle/input/*/*.npz")   # 1 cấp -> nhanh
    assert hits, "Chưa thấy .npz nào trong /kaggle/input — đã Add dataset phoenix-poses chưa?"
    POSE_DIR = os.path.dirname(hits[0])
n = len(glob.glob(os.path.join(POSE_DIR, "*.npz")))
print(f"pose_cache_dir = {POSE_DIR}  ({n} file .npz)")
s = open("configs/config.py", encoding="utf-8").read()
s = s.replace('pose_cache_dir: str = "/kaggle/input/phoenix-poses"',
              f'pose_cache_dir: str = "{POSE_DIR}"')
open("configs/config.py", "w", encoding="utf-8").write(s)

## **Cell 5 - CHẠY (theo cấu hình ở Cell 1)**

`matrix` → `run_all.py` (resumable, tự sinh bảng/hình ở cuối). `select` → `train_select.py`.
Chạy lại cell: `matrix` bỏ qua bước đã xong (marker `.done_*`); `select` cũng resume trong session.

In [ ]:
import subprocess, sys, re

# Ghi đè REWARD weights (mục 5 Cell 1) vào config.py TRƯỚC khi chạy — chỉ 4 reward cũ (không BERTScore).
# (config.py vừa clone lại ở Cell 3 nên luôn bắt đầu từ mặc định repo -> patch lại mỗi phiên.)
cfg_txt = open("configs/config.py", encoding="utf-8").read()
def _set_reward(txt, key, val):
    new, n = re.subn(rf"({key}: float = )[0-9.]+", rf"\g<1>{val}", txt, count=1)
    assert n == 1, f"Không tìm thấy '{key}: float =' trong config.py"
    return new
cfg_txt = _set_reward(cfg_txt, "reward_bleu_weight", REWARD_BLEU)
cfg_txt = _set_reward(cfg_txt, "reward_repetition_penalty", REWARD_REP)
cfg_txt = _set_reward(cfg_txt, "reward_length_penalty", REWARD_LEN)
open("configs/config.py", "w", encoding="utf-8").write(cfg_txt)
print(f"reward config -> w_bleu={REWARD_BLEU} w_rep={REWARD_REP} w_len={REWARD_LEN}")

if SCOPE == "matrix":
    cmd = [sys.executable, "run_all.py", "--subset", str(SUBSET), "--groups", GROUPS]
else:  # "select"
    cmd = [sys.executable, "train_select.py", "--mode", SELECT_MODE,
           "--encoder", ENCODER, "--algo", ALGO, "--subset", str(SUBSET), "--tag", TAG,
           "--w_bleu", str(REWARD_BLEU), "--w_rep", str(REWARD_REP), "--w_len", str(REWARD_LEN)]
print(">>", " ".join(cmd))
subprocess.run(cmd, check=True)

## **Cell 6 - Bảng kết quả của PHIÊN NÀY (những gì vừa train)**

In [ ]:
!python scripts/aggregate_results.py --work_dir /kaggle/working --out /kaggle/working/comparison_table
import pandas as pd
df = pd.read_csv("/kaggle/working/comparison_table.csv")
df.sort_values(["subset_pct", "encoder", "method"], na_position="last")

## **Cell 7 - Đóng gói kết quả NHẸ để tải về (không kèm checkpoint .pt)**

Chỉ gói `test_results.json` + `*_history.json` + bảng — đủ để gộp/vẽ lại ở Cell 8, dung lượng nhỏ.
Tải `results_bundle.tar.gz` ở tab **Output**, rồi upload thành 1 Kaggle Dataset (vd
`slt-results-<kiến trúc>`) để dùng ở bước gộp.

In [ ]:
import tarfile, glob, os
files = glob.glob("/kaggle/working/comparison_table.*")
for d in glob.glob("/kaggle/working/*_subset*"):
    if os.path.isdir(d):
        files += glob.glob(os.path.join(d, "test_results.json"))
        files += glob.glob(os.path.join(d, "*_history.json"))
out = "/kaggle/working/results_bundle.tar.gz"
with tarfile.open(out, "w:gz") as t:
    for f in files:
        t.add(f, arcname=os.path.relpath(f, "/kaggle/working"))
print(f"{len(files)} file -> {out}  (tải ở tab Output)")

## **Cell 8 - GỘP mọi phiên thành 1 bảng + hình (chạy khi đã có đủ kiến trúc)**

Add tất cả dataset `slt-results-*` (các `results_bundle.tar.gz` đã tải/ up) làm Input, rồi chạy
cell này: nó giải nén hết vào 1 work_dir chung, gộp bảng và vẽ hình báo cáo.

In [ ]:
import glob, tarfile, os
MERGE = "/kaggle/working/merged"
os.makedirs(MERGE, exist_ok=True)
bundles = glob.glob("/kaggle/input/**/results_bundle*.tar.gz", recursive=True)
for tb in bundles:
    with tarfile.open(tb) as t:
        t.extractall(MERGE)
print(f"Giải nén {len(bundles)} bundle vào {MERGE}")
!python scripts/aggregate_results.py --work_dir {MERGE} --out {MERGE}/comparison_table
!python scripts/make_report.py --work_dir {MERGE}
import pandas as pd
pd.read_csv(f"{MERGE}/comparison_table.csv").sort_values(["subset_pct", "encoder", "method"], na_position="last")